In [ ]:
# SWE 270P Assignment 2: Code Whiteboard Tutor
# Group members (Name - Student ID):
#   TODO: <Full Name> - <Student ID>
#   TODO: <Full Name> - <Student ID>

## Assignment 2: Code Whiteboard Tutor

Main goal: Use a multimodal Large Language Model (LLM) to build a UI application that allows users to upload a photo of their handwritten Python code and receive suggestions for code improvements.

The main functionalities that must be included are:
- Transcribing handwritten Python code into a code snippet
- Running static analysis and explaining bugs (if any) in natural language
- Suggesting bug fixes, improvements, or efficiency tweaks to the code snippet

**Workflow of the app:** upload image → **Transcribe Code** (image → code, editable) → **Analyze Code** (bug explanations, improvement suggestions, refined code).

**Extra credit: multiple programming languages.** Besides Python, the app supports Java, C, C++, C#, JavaScript, TypeScript, Go, and Rust. Choose the language in the dropdown, or leave it on *Auto-detect* and the model identifies the language itself.

In [ ]:
# Install required packages
%pip -q install gradio "google-genai>=1.50"

In [ ]:
# Import necessary libraries
import os
import time
from enum import Enum
from PIL import Image
from pydantic import BaseModel

import gradio as gr
from google import genai
from google.genai import types, errors

In [ ]:
# Load the Gemini API key.
# - Google Colab: add GEMINI_API_KEY in the Secrets tab (key symbol on the left navigation bar).
# - Local environment: set the GEMINI_API_KEY environment variable before starting Jupyter,
#   e.g. `export GEMINI_API_KEY=...`, so the key is never stored in this notebook.
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except ImportError:
    api_key = os.environ.get('GEMINI_API_KEY', '')

client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.5-flash-lite"

### ⚠️ A Note on Rate Limits

We're using `gemini-3.5-flash-lite` because it's a free-tier model, but rate limits still apply (and vary by model).

See the [Rate Limits docs](https://ai.google.dev/gemini-api/docs/rate-limits) for details. Three things to watch:

- **RPM (Requests Per Minute):** API calls allowed per 60-second window.
- **TPM (Tokens Per Minute):** input tokens allowed per 60-second window.
- **RPD (Requests Per Day):** total requests allowed per day (resets at midnight Pacific time).

Exceeding any one triggers a `429 RESOURCE_EXHAUSTED` error. If that happens, wait a few seconds and try again, and don't send too many requests in a short time.

The app makes exactly **one** request per button click, and `generate_with_retry` below waits and retries automatically when a `429` is returned.

### Helper functions

In [ ]:
def generate_with_retry(contents, config=None, max_retries=3):
    """
    Sends one request to the Gemini model, retrying if the free-tier rate limit is hit.

    A 429 RESOURCE_EXHAUSTED error means the RPM/TPM limit was exceeded; waiting a few
    seconds is usually enough, so we back off (5s, 10s, ...) and retry. Any other error,
    or a 429 on the last attempt, is raised to the caller.

    Parameters:
        contents (list): The request contents (PIL images and/or text prompts).
        config (types.GenerateContentConfig | None): Optional generation config.
        max_retries (int): Maximum number of attempts.

    Returns:
        The GenerateContentResponse returned by the model.
    """
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL_ID, contents=contents, config=config)
        except errors.APIError as e:
            if e.code != 429 or attempt == max_retries - 1:
                raise
            time.sleep(5 * (attempt + 1))


def strip_code_fences(text: str) -> str:
    """
    Removes a surrounding markdown code fence (```python ... ```) if the model added one,
    so the result can be shown in the code editor and saved directly as a .py file.
    Indentation inside the code is preserved.
    """
    lines = text.strip("\n").rstrip().splitlines()
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip().startswith("```"):
        lines = lines[:-1]
    return "\n".join(lines)

### Supported programming languages (extra credit)

In [ ]:
# Each supported language maps to:
#   (gr.Code syntax-highlighting name, or None when Gradio has no highlighter for it,
#    file extension used when saving the refined code)
LANGUAGES = {
    "Python": ("python", ".py"),
    "Java": (None, ".java"),
    "C": ("c", ".c"),
    "C++": ("cpp", ".cpp"),
    "C#": (None, ".cs"),
    "JavaScript": ("javascript", ".js"),
    "TypeScript": ("typescript", ".ts"),
    "Go": (None, ".go"),
    "Rust": (None, ".rs"),
}
AUTO_DETECT = "Auto-detect"

# Enum of the language names. It is used in the JSON response schemas below, so the model
# can only answer with one of the supported languages.
Language = Enum("Language", {name: name for name in LANGUAGES})


def language_instruction(language: str) -> str:
    """
    Returns the sentence that tells the model which language the code is in.
    For "Auto-detect" the model is asked to identify the language itself.
    """
    if language == AUTO_DETECT:
        return "First identify which programming language the code is written in and report it in the language field."
    return f"The code is written in {language}; report {language} in the language field."

### Step 1: Transcription (image → code)

In [ ]:
class Transcription(BaseModel):
    """Structured output schema for the transcription step."""
    language: Language   # programming language the code is written in
    code: str            # the code exactly as written in the image


# Prompt for transcription. The model must copy the code exactly as written (bugs included),
# because finding and fixing bugs is the job of the separate analysis step.
TRANSCRIBE_PROMPT = """
You are an expert at reading handwritten source code.
Transcribe the handwritten code in this image into plain text, exactly as written.
{language_instruction}

Rules:
- Copy the code faithfully. Do NOT fix bugs, logic errors, or style, even if they look wrong;
  the code will be reviewed in a separate step.
- Reproduce the indentation structure shown in the image using 4 spaces per level.
- Keep the original identifiers, operators, literals, comments, and line order.
- Ignore paper lines, margins, crossed-out text, and anything that is not code.
- If a handwritten character is ambiguous (e.g. `[` vs `C`, `:` vs `;`, `1` vs `l`, `x` vs `X`),
  choose the reading that is valid in the code's language and consistent with the rest of the code.
- Put only the code in the code field, with no explanations and no markdown code fences.
- If the image contains no code, set the code field to exactly: NO_CODE_FOUND
"""


def transcribe_image_code(image_file: Image.Image, language: str = AUTO_DETECT):
    """
    Step 1: Transcribes code from the input image using a multimodal LLM.

    The PIL image and the transcription prompt are sent together in one request
    (the google-genai SDK accepts PIL images directly). The reply is JSON matching the
    Transcription schema, so we get both the code and its programming language.
    Markdown fences are stripped from the code so it can be edited and analyzed as-is.

    Parameters:
        image_file (Image.Image): The uploaded image containing handwritten code.
        language (str): A key of LANGUAGES, or AUTO_DETECT to let the model identify it.

    Returns:
        A gr.update for the code editor: the transcribed code, with syntax highlighting
        and label set to the detected language.
    """
    if image_file is None:
        raise gr.Error("Please upload an image of handwritten code first.")

    try:
        prompt = TRANSCRIBE_PROMPT.format(language_instruction=language_instruction(language))
        config = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Transcription,
        )
        response = generate_with_retry([image_file, prompt], config=config)
        result = Transcription.model_validate_json(response.text)
    except Exception as e:
        raise gr.Error(f"Error during transcription: {str(e)}")

    code = strip_code_fences(result.code)
    if not code.strip() or code.strip() == "NO_CODE_FOUND":
        raise gr.Error("No code could be read from this image. Please try a clearer photo.")

    detected = result.language.value
    return gr.update(value=code, language=LANGUAGES[detected][0],
                     label=f"2. Transcribed code ({detected}, editable)")

### Step 2: Static analysis (bug explanations, improvements, refined code)

The model is asked to return JSON that matches the `CodeReview` schema, so the explanation and the refined code can be separated reliably instead of being parsed out of free text.

In [ ]:
class Bug(BaseModel):
    """One bug found by the static analysis."""
    location: str      # line number(s) / code fragment where the bug is
    explanation: str   # what goes wrong, in plain English
    fix: str           # how to fix it


class CodeReview(BaseModel):
    """Structured output schema for the analysis step."""
    language: Language        # programming language of the code
    summary: str              # what the code is intended to do
    bugs: list[Bug]           # empty if no bugs were found
    improvements: list[str]   # readability / robustness / efficiency suggestions
    fixed_code: str           # complete corrected code


# The refined code is saved as llm_fixed_code + the language's extension (e.g. llm_fixed_code.py),
# so Python results can be run against the JSON test cases below.
FIXED_CODE_BASENAME = "llm_fixed_code"


def format_review(review: CodeReview) -> str:
    """
    Converts the structured CodeReview into Markdown for the analysis panel of the UI:
    the language, a short summary, a numbered list of bugs (location, explanation, fix),
    and a list of improvement suggestions.
    """
    parts = [f"**Language:** {review.language.value}", "### What the code does", review.summary, "### Bugs found"]
    if review.bugs:
        for i, bug in enumerate(review.bugs, start=1):
            parts.append(f"**{i}. {bug.location}**: {bug.explanation}\n\n*Fix:* {bug.fix}")
    else:
        parts.append("No bugs found.")

    parts.append("### Suggested improvements")
    if review.improvements:
        parts.append("\n".join(f"- {s}" for s in review.improvements))
    else:
        parts.append("No further improvements suggested.")
    return "\n\n".join(parts)


def analyze_code(code_block: str, language: str = AUTO_DETECT):
    """
    Step 2: Performs static analysis using LLM on the transcribed code.

    The code is sent with a review prompt and the CodeReview JSON schema
    (response_mime_type="application/json"), so the reply always has separate fields for
    the language, the explanation, and the refined code. The refined code is also saved to
    llm_fixed_code.<ext> using the extension of the code's language.

    Parameters:
        code_block (str): The transcribed code snippet.
        language (str): A key of LANGUAGES, or AUTO_DETECT to let the model identify it.

    Returns:
        (1) The analysis text explaining bugs, improvements, or efficiency suggestions.
        (2) The refined version of the code snippet with the suggested fixes and enhancements
            (as a gr.update that also sets syntax highlighting for the language).
    """

    if not code_block or not code_block.strip():
        raise gr.Error("Please provide a valid code snippet.")

    try:
        prompt = f"""
        You are an expert software developer and code reviewer. Analyze the following code.
        {language_instruction(language)}
        It was transcribed from a photo of handwritten code and may contain logic bugs,
        syntax errors, or edge cases it does not handle.

        Perform a static analysis: reason about the code by reading it (it has not been run).
        Fill in these fields:
        1. language: The programming language of the code.
        2. summary: In 2-4 sentences, explain what the code is intended to do and how it works.
        3. bugs: Every bug you find (logic errors, off-by-one errors, wrong variables or indices,
           infinite loops or recursion, missing base cases, unhandled edge cases, null/undefined
           or memory errors, type errors, syntax errors).
           For each bug give its location (line number and the code on that line), explain in
           plain English what goes wrong (include a concrete input that triggers it when possible),
           and describe the fix. Use an empty list if the code is correct.
        4. improvements: Suggestions beyond the bug fixes for readability, style (the language's
           conventions, e.g. PEP 8 for Python), robustness, or efficiency (mention time/space
           complexity where relevant). Use an empty list if there is nothing worth changing.
        5. fixed_code: The complete corrected code, in the same language, with all bug fixes
           applied. Keep the same function name(s) and parameters so it can replace the original.
           Do not add example usage, tests, or print statements that were not in the original,
           and do not wrap it in markdown fences.

        Code to analyze:
        ```
        {code_block}
        ```
        """

        contents = [prompt]  # The LLM input is now a text prompt, not an image
        config = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CodeReview,
        )
        response = generate_with_retry(contents, config=config)

        # Parse the JSON reply into the schema, then build the two outputs
        review = CodeReview.model_validate_json(response.text)
        fixed_code = strip_code_fences(review.fixed_code)
        highlight, extension = LANGUAGES[review.language.value]
        with open(FIXED_CODE_BASENAME + extension, "w") as f:
            f.write(fixed_code + "\n")

        return format_review(review), gr.update(value=fixed_code, language=highlight,
                                                label=f"Refined code ({review.language.value})")

    except Exception as e:
        raise gr.Error(f"Error during analysis: {str(e)}")

### Gradio app

In [ ]:
# Gradio interface: upload -> transcribe (editable, so transcription mistakes can be corrected) -> analyze.
with gr.Blocks(title="Code Whiteboard Tutor") as app:
    gr.Markdown(
        "# Code Whiteboard Tutor\n"
        "Upload a photo of handwritten code, transcribe it, then run static analysis "
        "to get bug explanations, improvement suggestions, and a refined version of the code. "
        "Supports " + ", ".join(LANGUAGES) + "."
    )

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="1. Handwritten code image")
            language_input = gr.Dropdown(choices=[AUTO_DETECT, *LANGUAGES], value=AUTO_DETECT,
                                         label="Programming language")
            transcribe_btn = gr.Button("Transcribe Code", variant="primary")
        with gr.Column():
            code_input = gr.Code(language="python", label="2. Transcribed code (editable)", interactive=True)
            analyze_btn = gr.Button("Analyze Code", variant="primary")

    gr.Markdown("## 3. Static analysis")
    with gr.Row():
        with gr.Column():
            analysis_output = gr.Markdown()
        with gr.Column():
            fixed_code_output = gr.Code(language="python", label="Refined code", interactive=False)

    # Wire the buttons to the two steps (both use the selected language, or auto-detect it)
    transcribe_btn.click(transcribe_image_code, inputs=[image_input, language_input], outputs=code_input)
    analyze_btn.click(analyze_code, inputs=[code_input, language_input], outputs=[analysis_output, fixed_code_output])


if __name__ == "__main__":
    app.launch(share=True)

### Additional Resource: Running test cases on LLM-generated code.

In [ ]:
"""
Example of running test cases on LLM-generated code.
You do not need to follow this exact implementation for your code."""

import json
import importlib.util

def run_tests(filename_original, function_name, json_test_path):
    """
    Runs the JSON test cases against a function loaded from a Python file.

    Parameters:
        filename_original (str): Path to the .py file containing the function.
        function_name (str): Name of the function to test.
        json_test_path (str): Path to a JSON file of the form
            {"test_case": [{"input": [...], "expected": ...}, ...]}.

    Each test's "input" list is unpacked as the function's arguments; the result is
    compared with "expected" and a pass/fail line is printed per test.
    """
    # Load test cases
    with open(json_test_path, 'r') as f:
        test_cases = json.load(f)["test_case"]

    # Load the function from the file
    spec = importlib.util.spec_from_file_location(function_name, filename_original)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    func = getattr(module, function_name)

    # Run tests
    for idx, case in enumerate(test_cases):
        try:
            inputs = case["input"]
            expected = case["expected"]
            result = func(*inputs) if isinstance(inputs, (list, tuple)) else func(inputs)
            assert result == expected, f"input={inputs}, expected={expected}, got={result}"
            print(f"Test {idx+1} passed.")
        except Exception as e:
            print(f"Test {idx+1} failed: {e}")

In [ ]:
"""
After clicking "Analyze Code" in the app, the refined code is saved to llm_fixed_code.<ext>
(llm_fixed_code.py for Python). Run the matching test cases on it here.
run_tests loads the file as a Python module, so this works for Python code only.

Change function_name / json_test_path to match the image that was analyzed, e.g.
bucketsort, find_in_sorted, flatten, pascal, possible_change.
"""

run_tests(
    filename_original="llm_fixed_code.py",
    function_name="bucketsort",
    json_test_path="test_case_bucketsort.json"
)